# RAG Example: Loading Your Own Data into ChromaDB

This notebook shows how to load **any text data** into ChromaDB and query it.

We'll use the famous "Wear Sunscreen" speech (Chicago Tribune, 1997) as our example dataset.

Original source: [Salem Public Library — 1999 Yearbook](http://history.salem.lib.oh.us/yearbooks/1999/2facesplits/1999op_Part9.pdf)

## Step 1 — Read and chunk the text

The text file has the speech split into paragraphs (separated by blank lines).
We split on `\n\n` so each paragraph becomes one chunk in our database.

In [1]:
with open("../data/sunscreen_speech.txt") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print(f"{len(chunks)} chunks loaded\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:80]}...\n")

18 chunks loaded

Chunk 0: Ladies and gentlemen of the class of '99 ... Wear sunscreen. If I could offer yo...

Chunk 1: Enjoy the power and beauty of your youth; oh nevermind; you will not understand ...

Chunk 2: Don't worry about the future; or worry, but know that worrying is as effective a...

Chunk 3: Do one thing everyday that scares you. Sing. Don't be reckless with other people...

Chunk 4: Don't waste your time on jealousy. Sometimes you're ahead, sometimes you're behi...

Chunk 5: Remember compliments you receive. Forget the insults. If you succeed in doing th...

Chunk 6: Stretch. Don't feel guilty if you don't know what you want to do with your life....

Chunk 7: Get plenty of calcium. Be kind to your knees ... you'll miss them when they're g...

Chunk 8: Maybe you'll marry. Maybe you won't. Maybe you'll have children. Maybe you won't...

Chunk 9: Enjoy your body, use it everyway you can. Don't be afraid of it or what other pe...

Chunk 10: Read the directions even if you 

## Step 2 — Load into ChromaDB

In [2]:
import chromadb

client = chromadb.PersistentClient("../chroma")

# Delete collection if it already exists (so we can re-run this cell)
try:
    client.delete_collection("sunscreen_speech")
except Exception:
    pass

collection = client.create_collection(name="sunscreen_speech")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Added {collection.count()} chunks to the 'sunscreen_speech' collection")

/workspaces/ceu-ai-engineering-class/.venv/lib/python3.13/site-packages/chromadb/execution/expression/operator.py:239: SyntaxWarning: invalid escape sequence '\.'
  Key("email").regex(r".*@example\.com")
2026-03-02 13:36:32.572499655 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
/home/vscode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.3MiB/s]


Added 18 chunks to the 'sunscreen_speech' collection


## Step 3 — Query the collection

ChromaDB uses the query text to find the most semantically similar chunks.
This is the same pattern we use in `5_rag.ipynb` with the nutrition database.

In [3]:
results = collection.query(
    query_texts=["What does the author say about moving to another place?"],
    n_results=3,
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i + 1}:")
    print(doc)
    print()

Result 1:
Live in New York City once, but leave before it makes you hard. Live in northern California once, but leave before it makes you soft. Travel.

Result 2:
Understand that friends come and go, but for the precious few you should hold on. Work hard to bridge the gaps in geography and in lifestyle because the older you get, the more you need the people you knew when you were young.

Result 3:
Accept certain inalienable truths ... prices will rise, politicians will philander, you too will get old, and when you do, you'll fantasize that when you were young prices were reasonable, politicians were noble and children respected their elders. Respect your elders.

